In [ ]:
### Imports
import os
import time
import json
import logging
import shutil
import zipfile
from urllib.request import urlopen
from datetime import datetime, timedelta
#from concurrent.futures import ThreadPoolExecutor


### Custom functions
def extract_zip_recursively(folder_path, filename):
    file_path = os.path.join(folder_path, filename)

    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        filename_list = zip_ref.namelist()
        file_contains_zip = any(fname.endswith('.zip') for fname in filename_list)
        file_contains_non_zip = any(not fname.endswith('.zip') for fname in filename_list)

        if file_contains_zip:
            for fname in filename_list:
                if fname.endswith('.zip'):
                    zip_ref.extract(fname, folder_path)
                    extract_zip_recursively(folder_path, fname)

        if file_contains_non_zip:
            pass
        else:
            os.remove(file_path)

In [ ]:
### Set variables
source_name = 'bag'
host = 'https://service.pdok.nl/kadaster/adressen/atom/v1_0/downloads/lvbag-extract-nl.zip'
selection=['.zip']

In [ ]:
### Download file to temporary folder

download_folder = f"tmp/download/{source_name}"

# Connection contains multiple hosts: Elektra and Gas
file_name = host.split("/")[-1]

if os.path.exists(download_folder):
    shutil.rmtree(download_folder)
os.makedirs(download_folder, exist_ok=True)

# Download the zip file
file_path = os.path.join(download_folder, file_name)
with urlopen(host) as response, open(file_path, "wb") as f:
    shutil.copyfileobj(response, f)

In [ ]:
### Unpack files nested structure recursively

# Get list of downloaded files
file_list = sorted(os.listdir(download_folder))

# Get files based on selection criteria
selection_lower = [table_name.lower() for table_name in selection]
selected_files = [filename for filename in file_list if
                    any(table_name in filename.lower() for table_name in selection_lower)]

for filename in file_list:

    # Unpack .zip files inside .zip
    extract_zip_recursively(folder_path=download_folder, filename=filename)
